# Quantum kernel SVM — moons dataset

Uses PennyLane's `qml.kernels` module to compute a quantum kernel matrix
on the moons dataset. A kernel SVM (via sklearn SVC with precomputed
kernel) is trained on the quantum kernel to classify data.

The kernel measures overlap fidelity:
$K(x_1, x_2) = |\langle 0|U^\dagger(x_1) U(x_2)|0\rangle|^2$.
Data is rescaled to $[-\pi/2, \pi/2]$ for the angle embedding.

In [ ]:
import pennylane as qml
import numpy as np
from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler

## Setup

In [ ]:
N_FEATURES = 2
dev = qml.device("default.qubit", wires=N_FEATURES)

X_raw, Y_raw = make_moons(n_samples=60, noise=0.15, random_state=0)
X_train = X_raw[:40].astype(float)
Y_train = Y_raw[:40].astype(float)
X_test = X_raw[40:].astype(float)
Y_test = Y_raw[40:].astype(float)

scaler = MinMaxScaler(feature_range=(-np.pi / 2, np.pi / 2))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(f"train={len(X_train)}  test={len(X_test)}")

## Kernel circuit

In [ ]:
@qml.qnode(dev)
def kernel_circuit(x1, x2):
    qml.AngleEmbedding(x2, wires=range(N_FEATURES), rotation="Y")
    qml.adjoint(qml.AngleEmbedding)(x1, wires=range(N_FEATURES), rotation="Y")
    return qml.probs(wires=range(N_FEATURES))

def quantum_kernel(X1, X2):
    K = np.zeros((len(X1), len(X2)))
    for i, x1 in enumerate(X1):
        for j, x2 in enumerate(X2):
            probs = kernel_circuit(x1, x2)
            K[i, j] = probs[0]
    return K

## Compute kernel matrices and train SVM

In [ ]:
print("computing train kernel matrix...")
K_train = quantum_kernel(X_train, X_train)
print(f"  shape={K_train.shape}  range=[{K_train.min():.4f}, {K_train.max():.4f}]")

print("computing test kernel matrix...")
K_test = quantum_kernel(X_test, X_train)
print(f"  shape={K_test.shape}")

clf = SVC(kernel="precomputed", C=5.0)
clf.fit(K_train, Y_train)

## Evaluate

In [ ]:
train_acc = clf.score(K_train, Y_train)
test_acc = clf.score(K_test, Y_test)
print(f"train accuracy: {train_acc:.1%}")
print(f"test accuracy:  {test_acc:.1%}")

preds = clf.predict(K_test)
print()
for i in range(len(X_test)):
    mark = "" if preds[i] == Y_test[i] else "  <-- WRONG"
    print(f"  x=[{X_test[i, 0]:+.3f}, {X_test[i, 1]:+.3f}]  "
          f"pred={int(preds[i])}  true={int(Y_test[i])}{mark}")